$$\Large\boxed{\text{AME 5202 Deep Learning, Even Semester 2026}}$$

$$\large\text{Theme}: \underline{\text{building a regularized softmax classifier from scratch using efficient computations in PyTorch}}$$

---

We consider a softmax classfier, which is a 1-layer neural network or a 0-hidden layer neural network, for a batch comprising $b$ samples represented as the $b\times 784$-matrix $$\mathbf{X} = \begin{bmatrix}{\mathbf{x}^{(0)}}^\mathrm{T}\\{\mathbf{x}^{(1)}}^\mathrm{T}\\\vdots\\{\mathbf{x}^{(b-1)}}^\mathrm{T}\end{bmatrix}$$ with one-hot encoded true labels represented as the $b\times 10$-matrix (10 possible categories) $$\mathbf{Y}=\begin{bmatrix}{\mathbf{y}^{(0)}}^\mathrm{T}\\{\mathbf{y}^{(1)}}^\mathrm{T}\\\vdots\\{\mathbf{y}^{(b-1)}}^\mathrm{T}\end{bmatrix}.$$

The forward propagation for a generic sample in the batch seen as a $1\times784$-object $\mathbf{x}^\mathrm{T}$ with the bias feature $1$ added is presented below:

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}_B^\mathrm{T}}_{1\times785}=\begin{bmatrix}\mathbf{x}^\mathrm{T}&1\end{bmatrix}}&\rightarrow\boxed{\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times 10} = \underbrace{\mathbf{x}_B^\mathrm{T}}_{1\times785}\underbrace{{\mathbf{W}}}_{785\times10}}\rightarrow\boxed{\underbrace{{\mathbf{a}}^\mathrm{T}}_{1\times10}=\text{softmax}\left(\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times10}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^9-y_k\log\left(\hat{y}_k\right)}.
\end{align*}$$

The forward propagation for the same generic sample seen as a $784$-vector $\mathbf{x}$ with the bias feature $1$ added is presented below (note that the weight matrix has the same name $\mathbf{W}$ as above for simplicity even though it should show up as $\mathbf{W}^\mathrm{T}$):

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}_B}_{785}=\begin{bmatrix}\mathbf{x}\\1\end{bmatrix}}&\rightarrow\boxed{\underbrace{\mathbf{z}}_{10} = \underbrace{\mathbf{W}}_{10\times785}\underbrace{\mathbf{x}_B}_{785}}\rightarrow\boxed{\underbrace{\mathbf{a}}_{10}=\text{softmax}\left(\underbrace{\mathbf{z}}_{10}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^9-y_k\log\left(\hat{y}_k\right)}.\end{align*}$$

We will derive the update rule for the weights matrix $\mathbf{W}$ using the setup above.


The average crossentropy (CCE) loss for the batch is:$$\begin{align*}L &=\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]\\&=\frac{1}{b}\left[\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(0)}\log\left(\hat{y}^{(0)}_k\right)+\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(1)}\log\left(\hat{y}^{(1)}_k\right)+\cdots+\sum\limits_{k=0}^9{\color{yellow}-}y_k^{(b-1)}\log\left(\hat{y}^{(b-1)}_k\right)\right]\\&=\frac{1}{b}\left[{\color{yellow}-}{\mathbf{y}^{(0)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(0)}}\right)+{\color{yellow}-}{\mathbf{y}^{(1)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(1)}}\right)+\cdots+{\color{yellow}-}{\mathbf{y}^{(b-1)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(b-1)}}\right)\right].\end{align*}$$

The computational graph for the samples, each at a time treated as a $785$-vector, in the batch are presented below where the weights matrix has shape $10\times 785.$

$$\hspace{1.5in}\begin{align*}L_0\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(0)} &= \mathbf{a}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\hspace{0.25in}\begin{align*}L_1\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(1)} &= \mathbf{a}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\qquad\begin{align*} L_{2}\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(2)} &= \mathbf{a}^{(2)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(2)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\ldots\ldots\begin{align*} L_{b-1}\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(b-1)} &= \mathbf{a}^{(b-1)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(b-1)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\hspace{5in}$$

The gradient of the average batch loss w.r.t. the weights is:
$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &=\frac{1}{b}\left[\nabla_\mathbf{W}(L_0)+\nabla_\mathbf{W}(L_1)+\cdots+\nabla_\mathbf{W}(L_{b-1})\right]\end{align*}$$
which by chain rule can be written as:

$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &= \frac{1}{b}\left(\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(0)}\right) \times\nabla_{\mathbf{z}^{(0)}}\left(\hat{\mathbf{y}}^{(0)}\right)\times\nabla_{\hat{\mathbf{y}}^{(0)}}(L_0)\right]}_{\nabla_\mathbf{W}(L_0)}+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(1)}\right) \times\nabla_{\mathbf{z}^{(1)}}\left(\hat{\mathbf{y}}^{(1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(1)}}(L_1)\right]}_{\nabla_\mathbf{W}(L_1)}+\cdots+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(b-1)}\right) \times\nabla_{\mathbf{z}^{(b-1)}}\left(\hat{\mathbf{y}}^{(b-1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(b-1)}}(L_{b-1})\right]}_{\nabla_\mathbf{W}(L_{b-1})}\right)\\&=\frac{1}{b}\left(\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(0)}\right) \times\nabla_{\mathbf{z}^{(0)}}\left({\mathbf{a}}^{(0)}\right)\times\nabla_{\hat{\mathbf{y}}^{(0)}}(L_0)\right]}_{\nabla_\mathbf{W}(L_0)}+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(1)}\right) \times\nabla_{\mathbf{z}^{(1)}}\left({\mathbf{a}}^{(1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(1)}}(L_1)\right]}_{\nabla_\mathbf{W}(L_1)}+\cdots+\underbrace{\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(b-1)}\right) \times\nabla_{\mathbf{z}^{(b-1)}}\left(\hat{\mathbf{y}}^{(b-1)}\right)\times\nabla_{\hat{\mathbf{y}}^{(b-1)}}(L_{b-1})\right]}_{\nabla_\mathbf{W}(L_{b-1})}\right)\\&=\frac{1}{b}\sum_{i=0}^{b-1}\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right) \times\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)\right]\\&=\frac{1}{b}\sum_{i=0}^{b-1}\left[\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}_B}\right) \times\nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right)\right],\end{align*}$$
which can be written as

$$\small\begin{align*}\nabla_{\mathbf{W}}(L) &= \dfrac{1}{b}\displaystyle\sum_{i=0}^{b-1}\underbrace{\begin{bmatrix}\boxed{{\mathbf{x}^{(i)}_B}\ \pmb{0}\ \pmb{0}\ \ldots\ \pmb{0}}&&&&\\\\&\boxed{\pmb{0}\ {\mathbf{x}^{(i)}_B}\ \pmb{0}\ \ldots\ \pmb{0}}&&&\\&\hspace{1cm}\ddots&&&\\&&\hspace{-0.5cm}\ddots&&\\&&&\boxed{\pmb{0}\ \pmb{0}\ \ldots\ \pmb{0}\ {\mathbf{x}^{(i)}_B}}&\end{bmatrix}}_{\color{cyan}{\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right)=\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}_B}\right):\ 10\times785\times10}}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & \cdots & -a^{(i)}_0 a^{(i)}_9\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & \cdots & -a^{(i)}_1 a^{(i)}_9\\\vdots & \vdots & \ddots & \vdots\\-a^{(i)}_9 a^{(i)}_0 & -a^{(i)}_9 a^{(i)}_1 & \cdots & a^{(i)}_9 (1 - a^{(i)}_9)\end{bmatrix}}_{\color{cyan}{\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right) = \nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right):\ 10\times10}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0}\\-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\\vdots\\-\frac{y^{(i)}_9}{\hat{y}^{(i)}_9}\end{bmatrix}}_{\color{cyan}{\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)=\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right):\ 10\times1}}\end{align*}$$

The forward and backward propagation showing the gradient flow for a generic sample is shown below:

![](https://1drv.ms/i/c/37720f927b6ddc34/IQS3b-biQ4W9QpCtJzaZnyCoAQ8_r9i707rpOE1O9I0yntM?width=686&height=93)

$$\begin{align*}\nabla_{\mathbf{W}}(L) &=\dfrac{1}{b}\displaystyle\sum_{i=0}^{b-1}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & \cdots & -a^{(i)}_0 a^{(i)}_9\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & \cdots & -a^{(i)}_1 a^{(i)}_9\\\vdots & \vdots & \ddots & \vdots\\-a^{(i)}_9 a^{(i)}_0 & -a^{(i)}_9 a^{(i)}_1 & \cdots & a^{(i)}_9 (1 - a^{(i)}_9)\end{bmatrix}}_{\color{cyan}{=\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0} \\
-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\\vdots\\-\frac{y^{(i)}_9}{\hat{y}^{(i)}_9}\end{bmatrix}}_{\color{cyan}{=-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}}}{\mathbf{x}^{(i)}_B}^\mathrm{T}.\end{align*}$$

We can write the gradient in the following way for efficient coding purposes: $$\nabla_\mathbf{W}(L) = \frac{1}{b}\sum_{i=0}^{b-1}\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}\right]\left[-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}\right]{\mathbf{x}^{(i)}_B}^\mathrm{T}.$$


It can be seen that the gradient object has shape $(10\times 10)\times(10\times1)\times(1\times785)=10\times785,$ which is the same shape as the weights matrix $\mathbf{W}.$ However, our derivation here assumed that the samples are seen as column vectors of the data matrix. The original data matrix has the samples arranged as rows which corresponded to the weights matrix of shape $785\times10.$ In order to get the gradient w.r.t. that weights matrix, we have to transpose this expression resulting in the update $$\nabla_\mathbf{W}(L) = \frac{1}{b}\sum_{i=0}^{b-1}\underbrace{\mathbf{x}^{(i)}_B}_{\color{yellow}{785\times1}}\underbrace{\underbrace{\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]}_{\color{magenta}{\text{output side gradient of softmax layer: }1\times10}}\underbrace{\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\color{magenta}{\text{local gradient of softmax layer: }10\times10}}}_{\color{yellow}{\text{output side gradient of dense layer: }1\times10}}.$$



Note that when regularization is applied to the weights using a regularization strength $\lambda,$ then the regularization loss gets added to the data loss to give the total loss for the batch as follows: $$\begin{align*}L &= L_\text{data}+L_\text{reg}\\&=\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]+\lambda\left\{{\left[\mathbf{w}_0\right]_{:-1}}\cdot\left[\mathbf{w}_0\right]_{:-1}+{\left[\mathbf{w}_1\right]_{:-1}}\cdot\left[\mathbf{w}_1\right]_{:-1}+\cdots+{\left[\mathbf{w}_9\right]_{:-1}}\cdot\left[\mathbf{w}_9\right]_{:-1}\right\}.\end{align*}$$
Note that the last row of the weights matrix $\mathbf{W},$ which comprises the bias values, are not included in the regularization loss.

The gradient w.r.t. the weights matrix $\mathbf{W}$ now also includes the gradient w.r.t. the regularization loss as follows: $$\begin{align*}\nabla_\mathbf{W}(L)&=\nabla_\mathbf{W}\left(\frac{1}{b}\left[L_0+L_1+\cdots+L_{b-1}\right]\right)+\lambda\nabla_\mathbf{W}\left(L_\text{reg}\right)\\&=\underbrace{\frac{1}{b}\sum_{i=0}^{b-1}{\mathbf{x}^{(i)}_B}\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\text{data gradient}}+\lambda\underbrace{\begin{bmatrix}2{\mathbf{w}_{0}}&2{\mathbf{w}_{1}}&2{\mathbf{w}_{2}}&\ldots&2{\mathbf{w}_{9}}\\\pmb{0}&\pmb{0}&\pmb{0}&\ldots&\pmb{0}\end{bmatrix}}_{\text{regularization gradient}}.\end{align*}$$


---

---

Load libraries

---

In [ ]:
## Load libraries
import numpy as np
import sys
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix

---

The **MNIST (Modified National Institute of Standards and Technology) dataset** is one of the most widely used benchmark datasets in machine learning and computer vision.

It contains:

- **70,000 grayscale images**
- **28 × 28 pixels** per image
- **10 classes (digits 0–9)**
- Split into **60,000 training** and **10,000 test** images

Each image shows a centered handwritten digit as in the example below:

![mnist](https://1drv.ms/i/c/37720f927b6ddc34/IQQ6UqroXhRjSp1y9MW00bT_AR8-ENNoQV9lkr8uUoFpf7Q)

We will load the MNIST dataset and transform the images (normalize pixel values from 0-1 and flatten) using PyTorch.


---

In [ ]:
# Define PyTorch transformation pipeline for the MNIST dataset
transform = transforms.Compose([
    # Converts [28x28] gradyscale image → [1,28,28] tensor normalized to [0, 1]
    transforms.ToTensor(),
    # Flatten to [784]
    transforms.Lambda(lambda x: torch.flatten(x))
])

# Load train dataset
train_dataset = datasets.MNIST(
    root = './data',
    train = True,
    download = True,
    transform = transform
)

# Load test dataset
test_dataset = datasets.MNIST(
    root = './data',
    train = False,
    download = True,
    transform = transform
)

num_samples = len(train_dataset), len(test_dataset)
num_features = train_dataset[0][0].shape[0]
num_labels = torch.unique(train_dataset.targets).numel()

print('MNIST set')
print('---------------------')
print(f'Number of training samples = {num_samples[0]}, test samples = {num_samples[1]}')
print(f'Number of features = {num_features}')
print(f'Number of output labels = {num_labels}')


---

A generic layer class with forward and backward methods

----

In [ ]:
class Layer():
  def __init__(self):
    self.input = None
    self.output = None

  def forward(self, input):
    raise NotImplementedError("Forward propagation not implemented")

  def backward(self, output_gradient, learning_rate):
    raise NotImplementedError("Backward propagation not implemented")

---


Categorical crossentropy (CCE) loss and its gradient for the batch samples.

---

In [ ]:
## Define the loss function and its gradient
class CategoricalCrossEntropyLoss:
  def __init__(self, num_labels):
    self.num_labels = num_labels
    self.Y_onehot = None

  def forward(self, Y, Yhat):
    self.Y_onehot = F.one_hot(Y, num_classes = self.num_labels).float()
    return torch.mean(-torch.log(torch.sum(self.Y_onehot*Yhat, dim = -1)))

  def backward(self, Y, Yhat):
    return -self.Y_onehot/Yhat

---


Softmax activation layer class


---

In [ ]:
## Softmax activation layer class
class Softmax(Layer):
  def __init__(self):
    super(Softmax, self).__init__()

  def forward(self, input):
    self.input = input
    # Calculate softmax(input)
    self.output = F.softmax(self.input, dim = -1)
    return self.output

  def backward(self, output_gradient, learning_gradient = None):
    I = torch.eye(self.output.shape[1])
    softmax_local_gradient = (I - self.output.unsqueeze(-1)) * self.output.unsqueeze(-2)
    # Calculate gradient flowing back on the input side of the softmax layer
    input_gradient =  torch.einsum('ij, ijk -> ik', output_gradient, softmax_local_gradient)
    return input_gradient

---


Dense layer class

---

In [ ]:
## Dense layer class
class Dense(Layer):
  def __init__(self, input_size, output_size, reg_strength = 0.0):
    super(Dense, self).__init__()
    self.weights = torch.randn(input_size+1, output_size, requires_grad = True) * 0.01
    with torch.no_grad():
      self.weights[-1].fill_(0.01) # set all bias values equal to 0.01
    self.reg_strength = reg_strength
    self.reg_loss = None


  def forward(self, input):
    # Adding the bias feature 1 as the last column
    self.input = torch.hstack([input, torch.ones(input.shape[0], 1)]) 
    self.output= self.input @ self.weights
    self.reg_loss = self.reg_strength * torch.sum(self.weights[:-1] * self.weights[:-1])
    return self.output, self.reg_loss

  def backward(self, output_gradient, learning_rate = 1e-02):
    # Calculate local gradient w.r.t. weights
    weights_gradient = (1/output_gradient.shape[0]) * (torch.einsum('ij,ik->jk', self.input, output_gradient))
    # Add the regularization gradient
    weights_gradient += self.reg_strength * 2 * torch.vstack([self.weights[:-1], torch.zeros(1, self.weights.shape[1])])
    # Update weights for dense layer
    with torch.no_grad():
      self.weights += learning_rate * (-weights_gradient)

---



The neural network class for the softmax classifier.


---

In [ ]:
class NeuralNetwork:
  def __init__(self,
               num_features,
               num_labels,
               loss_fn,
               learning_rate = 1e-02,
               reg_strength = 0.0):
    self.num_features = num_features
    self.num_labels = num_labels
    self.loss_fn = loss_fn
    self.learning_rate = learning_rate
    self.reg_strength = reg_strength
    self.reg_loss = None
    # Architecture
    self.layers = [
        Dense(self.num_features, self.num_labels),
        Softmax()
        ]

  # Forward propagation
  def forward(self, x):
    self.reg_loss = 0.0
    for layer in self.layers:
      if hasattr(layer, 'weights'):
        x, reg_loss = layer.forward(x)
        self.reg_loss += reg_loss
      else:
        x = layer.forward(x)  
    return x

  # Backward propagation
  def backward(self, gradient):
    for layer in reversed(self.layers):
      gradient = layer.backward(gradient, self.learning_rate)

---


Train the softmax classifier.


---

In [ ]:
# Initialize model and optimizer
learning_rate = 1e-01
nepochs = 350
batch_size = 100
reg_strength = 0.1

# Choose appropriate loss function
loss_fn = CategoricalCrossEntropyLoss(num_labels)

# Data loader for batch processing
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

model = NeuralNetwork(num_features, num_labels, loss_fn, learning_rate, reg_strength)

# Create empty list to store training and test losses over each epoch
train_loss = [None]*nepochs
test_loss = [None]*nepochs

for epoch in range(nepochs):
  loss = 0.0

  for x_batch, y_batch in train_loader:
    # Forward pass
    predictions = model.forward(x_batch)
    loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

    # Backward pass
    loss_gradient = model.loss_fn.backward(y_batch, predictions)
    model.backward(loss_gradient)

  train_loss[epoch] = loss.detach().numpy() / len(train_loader)

  # Test loss calculation
  loss = 0.0

  with torch.no_grad():
    for x_batch, y_batch in test_loader:
      predictions = model.forward(x_batch)
      loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

  test_loss[epoch] = loss.detach().numpy() / len(test_loader)

  print(f"Epoch {epoch + 1}/{nepochs}, Train Loss: {train_loss[epoch]:.4f}, Test Loss: {test_loss[epoch]:.4f}")

---

Plot training and test loss vs. epoch

---

In [ ]:
## Plot train and test loss as a function of epoch:
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
fig.tight_layout(pad = 4.0)
ax.plot(train_loss, 'b', label = 'Train')
ax.plot(test_loss, 'r', label = 'Test')
ax.set_xlabel('Epoch', fontsize = 12)
ax.set_ylabel('Loss value', fontsize = 12)
ax.legend()
ax.set_title('Loss vs. Epoch', fontsize = 14);

---

Assess model performance on test data

---

In [ ]:
all_preds = []
all_true = []

with torch.no_grad():
  for x_batch, y_batch in test_loader:
    outputs = model.forward(x_batch)
    preds = torch.argmax(outputs, dim = 1)
    all_preds.append(preds)
    all_true.append(y_batch)

ypred = torch.cat(all_preds)
ytrue = torch.cat(all_true)

accuracy = (ypred == ytrue).float().mean() * 100
print(f'Accuracy on test data = {accuracy:3.2f}')

print(f'Confusion matrix:\n{confusion_matrix(ytrue.cpu(), ypred.cpu())}')

---

Plot a random test sample with its predicted label printed above the plot.

---

In [ ]:
## Plot a random test sample with its predicted label printed above the plot
test_index = np.random.choice(len(test_dataset))
fig, ax = plt.subplots(1, 1, figsize = (2, 2))
print(
    f"Image {'classified correctly' if ypred[test_index] == ytrue[test_index] else 'classified incorrectly'} "
    f"as {ypred[test_index]}"
)
ax.imshow(test_dataset[test_index][0].detach().numpy().reshape(28, 28), cmap = 'gray');